# Setup

In [ ]:
%%capture
%pip install --quiet pydantic-ai nest_asyncio
# Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Set your OpenRouter API key
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass
if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

MODEL = OpenAIChatModel(
    'openai/gpt-4o-mini',
    # 'openai/gpt-5.4-mini',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
)

# Extracting information without structured outputs


In [ ]:
from pydantic_ai import Agent

abstract = """
In this study, we introduce a novel deep learning approach for predicting protein-protein interactions (PPIs) in Saccharomyces cerevisiae.
Our method leverages graph neural networks to capture complex molecular interactions and achieves an AUC-ROC score of 0.92 on the independent test set.
The model outperforms traditional machine learning methods and provides interpretable insights into key interacting residues.
"""

plain_agent = Agent(MODEL, system_prompt="Extract the title, author, organism, method, and metric from the provided abstract.")
raw_response = plain_agent.run_sync(user_prompt=f'Abstract: {abstract}')

print(raw_response.output) # Hard to parse and to work with!

In [ ]:
lines = raw_response.output.split('\n')
metric_line = next((line for line in lines if line.startswith('Metric:')), None)

if metric_line:
    parsed_metric = metric_line.replace('Metric:', '').strip()
    print(f"Parsed Metric from raw response: {parsed_metric}")
else:
    print("Metric not found in raw response.")

# Structured output with PydanticAI

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel

class PaperSummary(BaseModel):
    title: str | None
    author: str | None
    organism: str | None
    method: str | None
    metric: float | None

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=PaperSummary)
result = agent.run_sync(f"Abstract to parse: {abstract}")
structured_output = result.output
print(type(structured_output))
print(structured_output.metric) #Actually returns a float number

In [ ]:
print(structured_output.model_dump_json(indent=2))
# Output can still be hallucinated or output UNKNOWN instead of None

# Exercise

In [ ]:
#TODO create an LLM-call that will use an abstract to extract: biomedical problem addressed, ml approach, input data, and model output
text = """
In this study, we propose iDNA-ABF, a multi-scale deep biological language learning model
that enables the interpretable prediction of DNA methylations based on genomic sequences only.
Benchmarking comparisons show that iDNA-ABF outperforms state-of-the-art methods for
different methylation predictions. By integrating an interpretable analysis mechanism, the model
helps map important sequential determinants to downstream biological functions.
"""


# Validators

Validators allows you to define custom validation logic for your structured outputs.

In [ ]:
new_abstract = """
In this study, we propose iDNA-ABF, a multi-scale deep biological language learning model
that enables the interpretable prediction of DNA methylations based on genomic sequences only.
Benchmarking comparisons show that iDNA-ABF outperforms state-of-the-art methods for
different methylation predictions, achieving an accuracy of 92.6%. By integrating an interpretable analysis mechanism, the model
helps map important sequential determinants to downstream biological functions.
"""

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent

class MethodPerformance(BaseModel):
    method_name: str | None
    accuracy: float | None

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=MethodPerformance)
non_validated_result = agent.run_sync(user_prompt=f"Abstract to parse: {new_abstract}")
non_validated_result.output

In [ ]:
my_paper = MethodPerformance(method_name="BioDNA", accuracy=0.950)

compared = max(my_paper.accuracy, non_validated_result.output.accuracy)
print(f"Best solution accuracy: {compared}") # Wrong! 0.950 > 0.926

Let's validate that the accuracy is between 0 and 1. **Always**.

In [ ]:
from pydantic_ai import Agent, ModelRetry

validated_agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=MethodPerformance, retries=2)

@validated_agent.output_validator
def validate_accuracy(output: MethodPerformance) -> MethodPerformance:
    if not (0 <= output.accuracy <= 1):
        raise ModelRetry("Accuracy must be between 0 and 1.") # Feeds the error back to the agent to retry
    return output

validated_result = validated_agent.run_sync(user_prompt=f"Abstract to parse: {new_abstract}")
validated_result.output

In [ ]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)
            print(f'    └─ {kind}: {snippet}')

In [ ]:
pretty_print_trace(validated_result)

In [ ]:
#TODO show exceptions for max retries 
# reached? Maybe if asked or show by changing the retries to 0

# Exercise

Given a list of papers, extract the DOIs. The DOIs must be valid.

In [ ]:
PAPERS = [
    {
        "title": "Agentomics: An Agentic System that Autonomously Develops "
                "State-of-the-art Solutions for Biomedical Machine Learning Tasks",
        "doi": "https://doi.org/10.64898/2026.01.27.702049",
        "abstract": (
            "Automated machine learning tools still require substantial human "
            "oversight when applied to biomedical data. We present Agentomics, an "
            "autonomous LLM-driven framework that implements multiple modeling "
            "strategies, inserts validation checkpoints, and produces "
            "deployment-ready models without human intervention. Across 20 datasets "
            "spanning protein engineering, drug discovery, and regulatory genomics, "
            "Agentomics outperforms competing agentic systems and reaches "
            "state-of-the-art results on 11 of 20 established benchmarks."
        ),
    },
    {
        "title": "A graph neural network for predicting protein–protein interactions "
                "in Saccharomyces cerevisiae",
        "abstract": (
            "We introduce a graph neural network for predicting protein–protein "
            "interactions from sequence and structural features in Saccharomyces "
            "cerevisiae. The model captures higher-order molecular context through "
            "message passing over an interaction graph and achieves an AUC-ROC of "
            "0.92 on an independent test set, outperforming classical baselines. "
            "An interpretability analysis highlights the residues that most "
            "influence each predicted interaction. DOI: 10.3390/molecules27186135"
        ),
    },
    {
        "title": "Masked-language pretraining improves DNA methylation prediction",
        "doi": None,
        "abstract": (
            "We propose a multi-scale biological language model that predicts DNA "
            "methylation from genomic sequence alone. Benchmarking against "
            "state-of-the-art methods across several methylation types, our approach "
            "achieves an accuracy of 92.6% while remaining interpretable, mapping "
            "salient sequence determinants to downstream biological functions."
        ),
    },
        {
        "title": "Masked-language pretraining improves DNA methylation prediction",
        "doi": "10.9999/does-not-exist",
        "abstract": (
            "We propose a multi-scale biological language model that predicts DNA "
            "methylation from genomic sequence alone. Benchmarking against "
            "state-of-the-art methods across several methylation types, our approach "
            "achieves an accuracy of 92.6% while remaining interpretable, mapping "
            "salient sequence determinants to downstream biological functions."
        ),
    },
  ]


In [ ]:
class FormattedDois(BaseModel): # output type the agent will return
    dois: list[str]

# Define the agent to extract valid DOI from the papers

In [ ]:
import httpx

def doi_exists(output: str) -> bool: # Needs to be registered to the agent as a validator
    for doi in output.dois:
        doi = doi.strip().removeprefix("https://doi.org/")
        resp = httpx.get(f"https://api.crossref.org/works/{doi}", timeout=10)
        if resp.status_code != 200:
            pass # Define and return the error message to be raised when the DOI does not exist
    return output


def format_doi(output: str) -> bool: # Needs to be registered to the agent as a validator
    for doi in output.dois:
        doi = doi.strip()
        if not doi.startswith("https://doi.org/"):
            pass # Define and return the error message to be raised when the DOI does not exist
    return output

In [ ]:
# Run the agent to extract DOIs from the abstracts